# TCCT v3.2.1 — Pipeline portátil, retomável e cientificamente auditável

**Projeto:** revisão do Paper 142 — estimativa de altura do dossel e biomassa em Paracou com fusão Landsat-5, PolInSAR L/P e LiDAR.

Esta versão mantém o par interferométrico definitivo **tropi0506–tropi0603**, usa as grades tridimensionais `.grille` da ONERA para o back-geocoding e preserva os resultados já produzidos em `Resultados/TCCT_v3_2_SIBGRAPI`.

## Como usar

1. Coloque este notebook e o script `Pipeline_TCCT_v3_2_1_SIBGRAPI_REV_GRILLE3D_RETOMAVEL.py` dentro de `SIBGRAPI_REV/Codigo`.
2. Execute sempre os Passos **00 e 01** ao abrir um novo kernel ou mudar de computador.
3. Os Passos 02–19 só devem ser repetidos quando houver alteração deliberada nos dados ou na configuração.
4. O Passo 20 agora identifica explicitamente `seed`, `fold`, progresso e estimativa de tempo.
5. Ao interromper o Passo 20, as combinações completas são puladas na retomada. Nas execuções iniciadas pela v3.2.1, a fase atual também retoma da última época concluída; se a interrupção ocorrer no meio de uma época, apenas essa época é refeita.

> **Importante:** os checkpoints de retomada não alteram a função de perda, a arquitetura, os folds ou os hiperparâmetros científicos. Eles apenas preservam o estado computacional do treino.

## Passo 00 — Detecção portátil da raiz e carregamento do módulo

**Objetivo.** Localizar de forma inequívoca a pasta `SIBGRAPI_REV` em macOS, Windows ou Linux e carregar o script científico correto.

**Por que existe.** Caminhos absolutos diferentes entre computadores podem fazer o pipeline ler ou gravar dados no local errado. O carregador procura a variável `TCCT_SIBGRAPI_REV_DIR`, o diretório atual e seus pais, volumes do macOS e unidades do Windows. Se encontrar mais de uma cópia válida, interrompe em vez de escolher silenciosamente.

**Critério para avançar.** A saída deve mostrar o sistema operacional, a raiz detectada, a versão `3.2.1-SIBGRAPI-REV-GRILLE3D-RETOMAVEL`, o dataset v6 e a pasta de resultados correta.

In [11]:
# PASSO 00 — CARREGAMENTO PORTÁTIL (macOS / Windows / Linux)
from pathlib import Path
import importlib.util
import os
import platform
import string
import sys

NOME_SCRIPT = "Pipeline_TCCT_v3_2_1_SIBGRAPI_REV_GRILLE3D_RETOMAVEL.py"
NOME_RAIZ = "SIBGRAPI_REV"
VAR_RAIZ = "TCCT_SIBGRAPI_REV_DIR"


def _raiz_valida(caminho: Path) -> bool:
    caminho = caminho.expanduser()
    return (
        caminho.is_dir()
        and caminho.name == NOME_RAIZ
        and (caminho / "Codigo" / NOME_SCRIPT).is_file()
    )


def _candidatos_a_partir_do_cwd() -> list[Path]:
    cwd = Path.cwd().resolve()
    candidatos = [cwd, cwd / NOME_RAIZ]
    candidatos.extend(cwd.parents)
    candidatos.extend(p / NOME_RAIZ for p in cwd.parents)
    return candidatos


def _candidatos_windows() -> list[Path]:
    candidatos = []
    usuario = os.environ.get("USERNAME", "")

    for letra in string.ascii_uppercase:
        drive = Path(f"{letra}:/")
        try:
            if not drive.exists():
                continue
        except OSError:
            continue

        candidatos.extend([
            drive / NOME_RAIZ,
            drive / "PROJECT_VOLUME" / NOME_RAIZ,
            drive / "TCCT_Geral" / NOME_RAIZ,
            drive / "TCCT_Geral" / "Perplexity" / NOME_RAIZ,
        ])

        if usuario:
            candidatos.extend([
                drive / "Users" / usuario / NOME_RAIZ,
                drive / "Users" / usuario / "Desktop" / NOME_RAIZ,
                drive / "Users" / usuario / "Documents" / NOME_RAIZ,
            ])

        # Busca controlada, no máximo dois níveis abaixo da raiz da unidade.
        for padrao in (f"*/{NOME_RAIZ}", f"*/*/{NOME_RAIZ}"):
            try:
                candidatos.extend(drive.glob(padrao))
            except (OSError, PermissionError):
                pass

    return candidatos


def localizar_raiz_projeto() -> Path:
    candidatos = []

    # 1. Variável de ambiente, somente se apontar para uma raiz válida.
    valor_ambiente = os.environ.get(VAR_RAIZ)
    if valor_ambiente:
        candidatos.append(Path(valor_ambiente))

    # 2. Diretório atual e seus pais.
    candidatos.extend(_candidatos_a_partir_do_cwd())

    # 3. Candidatos específicos do sistema.
    sistema = platform.system()
    if sistema == "Darwin":
        candidatos.extend([
            Path("<LOCAL_PATH>"),
            Path.home() / NOME_RAIZ,
        ])
    elif sistema == "Windows":
        candidatos.extend(_candidatos_windows())
    else:
        usuario = os.environ.get("USER", "")
        candidatos.extend([
            Path("<LOCAL_PATH>"),
            Path("/media") / usuario / "PROJECT_VOLUME" / NOME_RAIZ,
            Path.home() / NOME_RAIZ,
        ])

    # Elimina duplicatas preservando a ordem.
    vistos = set()
    candidatos_unicos = []
    for candidato in candidatos:
        try:
            chave = str(candidato.expanduser().resolve(strict=False))
        except OSError:
            chave = str(candidato.expanduser())
        if chave not in vistos:
            vistos.add(chave)
            candidatos_unicos.append(candidato)

    encontrados = []
    for candidato in candidatos_unicos:
        try:
            candidato = candidato.expanduser().resolve(strict=False)
            if _raiz_valida(candidato):
                encontrados.append(candidato)
        except OSError:
            continue

    if len(encontrados) == 1:
        return encontrados[0]

    if len(encontrados) > 1:
        print("Foram encontradas múltiplas cópias válidas do projeto:")
        for i, caminho in enumerate(encontrados, start=1):
            print(f"  [{i}] {caminho}")
        raise RuntimeError(
            "Defina explicitamente TCCT_SIBGRAPI_REV_DIR para evitar abrir "
            "a cópia errada do projeto."
        )

    raise FileNotFoundError(
        "Não foi possível localizar automaticamente SIBGRAPI_REV.\n"
        "Defina a variável de ambiente TCCT_SIBGRAPI_REV_DIR com o caminho "
        "completo da pasta do projeto e execute novamente o Passo 00."
    )


RAIZ_PROJETO = localizar_raiz_projeto()
os.environ[VAR_RAIZ] = str(RAIZ_PROJETO)

SCRIPT_PIPELINE = RAIZ_PROJETO / "Codigo" / NOME_SCRIPT

NOME_MODULO = "tcct_pipeline_v3_2_1_grille3d_retomavel"
sys.modules.pop(NOME_MODULO, None)
spec = importlib.util.spec_from_file_location(NOME_MODULO, SCRIPT_PIPELINE)
if spec is None or spec.loader is None:
    raise ImportError(f"Não foi possível criar o módulo a partir de {SCRIPT_PIPELINE}")

tcct = importlib.util.module_from_spec(spec)
sys.modules[NOME_MODULO] = tcct
spec.loader.exec_module(tcct)

if tcct.VERSAO_PIPELINE != "3.2.1-SIBGRAPI-REV-GRILLE3D-RETOMAVEL":
    raise RuntimeError(tcct.VERSAO_PIPELINE)

tcct.CFG.dados.base_dir = str(RAIZ_PROJETO)

print("Sistema:", platform.system())
print("Raiz detectada:", RAIZ_PROJETO)
print("Pipeline:", tcct.VERSAO_PIPELINE)
print("Dataset:", tcct.CFG.dados.dir_dataset_v2)
print("Resultados:", tcct.CFG.dados.dir_resultados_v2)
print("Back-geocoding:", tcct.CFG.dados.dir_backgeo_grille3d)


TCCT pipeline 3.2.1-SIBGRAPI-REV-GRILLE3D-RETOMAVEL
Sistema=Windows | Python=3.13.13 | PyTorch=2.11.0+cu128
Dispositivo=cuda | Base=<LOCAL_PATH>
Sistema: Windows
Raiz detectada: <LOCAL_PATH>
Pipeline: 3.2.1-SIBGRAPI-REV-GRILLE3D-RETOMAVEL
Dataset: <LOCAL_PATH>
Resultados: <LOCAL_PATH>
Back-geocoding: <LOCAL_PATH>


## Passo 01 — Configuração científica canônica

**Objetivo.** Fixar as opções que definem o experimento: par 0506–0603, buffer espacial principal, método vertical do DTM, back-geocoding sem extrapolação e política de sobrescrita.

**Por que existe.** Um pipeline reprodutível não deve depender de valores residuais deixados na memória do notebook. Esta célula reaplica a configuração após cada novo carregamento do módulo.

**Critério para avançar.** Não modifique o par interferométrico, os layouts binários ou a interpolação para tentar melhorar métricas. Mudanças metodológicas devem ser registradas como novos experimentos.

In [12]:
# PASSO 01 — CONFIGURAÇÃO GLOBAL
# Não altere o par, os layouts ou o método de interpolação.
tcct.CFG.dados.sobrescrever = False
tcct.CFG.dados.buffer_blocos = 0
tcct.CFG.dados.buffers_a_auditar = (0, 1)
tcct.CFG.dados.backgeo_modo_vertical_dtm = "auto_roi"
tcct.CFG.dados.backgeo_fracao_geometrica_minima_patch = 1.0
tcct.CFG.dados.backgeo_interpolacao_slc = "bilinear_complexa"
tcct.CFG.dados.aquisicao_mestre = "tropi0506"
tcct.CFG.dados.aquisicao_escrava = "tropi0603"
tcct.CFG.dados.aquisicoes_ortho = ("tropi0506", "tropi0603")
tcct.CFG.dados.bandas = ("SAR_L", "SAR_P")

print("Par:", tcct.CFG.dados.aquisicao_mestre, tcct.CFG.dados.aquisicao_escrava)
print("Buffer principal:", tcct.CFG.dados.buffer_blocos)
print("Modo vertical DTM:", tcct.CFG.dados.backgeo_modo_vertical_dtm)
print("Fração geométrica mínima por patch:", tcct.CFG.dados.backgeo_fracao_geometrica_minima_patch)


Par: tropi0506 tropi0603
Buffer principal: 0
Modo vertical DTM: auto_roi
Fração geométrica mínima por patch: 1.0


## Passo 02 — Limpeza controlada de produtos obsoletos

**Objetivo.** Remover apenas produtos derivados da geometria antiga `ROI + KDTree`, datasets v4/v5 e resultados incompatíveis.

**Por que existe.** Arquivos antigos podem ser encontrados por buscas automáticas e contaminar o processamento atual. A limpeza preserva dados brutos, LiDAR canônico, Landsat, relatórios de auditoria e a grade geoidal.

**Execução segura.** Primeiro use `CONFIRMAR_LIMPEZA=False` para listar. A exclusão real deve ocorrer uma única vez com `True`.

**Critério para avançar.** O relatório de limpeza deve ficar salvo em `Auditoria`; nunca apague `Dados_Brutos`.

In [ ]:
# PASSO 02 — LIMPEZA CONTROLADA
EXECUTAR_PASSO_02 = False
CONFIRMAR_LIMPEZA = False

if EXECUTAR_PASSO_02:
    REMOVIDOS_OU_LISTADOS = tcct.remover_produtos_obsoletos_pre_grille(
        tcct.CFG.dados,
        confirmar=CONFIRMAR_LIMPEZA,
    )
else:
    print("Passo 02 não solicitado.")


## Passo 03 — Estrutura isolada do projeto

**Objetivo.** Criar e validar a árvore `Codigo`, `Dados_Brutos`, `Dados_Processados`, `Dataset`, `Resultados`, `Auditoria` e `Logs`.

**Por que existe.** A revisão deve ser reproduzível sem reutilizar silenciosamente produtos de `TCCT_Geral`, `Perplexity` ou versões anteriores.

**Critério para avançar.** A raiz final precisa se chamar exatamente `SIBGRAPI_REV`, e o marcador do projeto deve apontar para a mesma raiz exibida no Passo 00.

In [ ]:
# PASSO 03 — ESTRUTURA ISOLADA DO PROJETO
EXECUTAR_PASSO_03 = False
if EXECUTAR_PASSO_03:
    ESTRUTURA = tcct.executar_etapa(
        "Passo 03 — Estrutura isolada do projeto",
        tcct.preparar_estrutura_projeto,
        tcct.CFG.dados, True,
    )
else:
    print("Passo 03 não solicitado.")


## Passo 04 — Inventário integral do acervo

**Objetivo.** Catalogar arquivos originais, tamanhos, extensões, bandas, aquisições e hashes.

**Por que existe.** O inventário cria rastreabilidade entre os dados usados e os resultados. Ele também evita que um arquivo com nome semelhante, mas conteúdo diferente, seja selecionado.

**Saídas.** CSVs e relatórios em `Auditoria/Inventario_TropiSAR`.

**Critério para avançar.** As `.grille` devem ser classificadas como `grille`, os 16 SLCs como `slc` e os arquivos Ha-P como altura de ambiguidade.

In [ ]:
# PASSO 04 — INVENTÁRIO INTEGRAL DO ACERVO
EXECUTAR_PASSO_04 = False
if EXECUTAR_PASSO_04:
    INVENTARIO = tcct.executar_etapa(
        "Passo 04 — Inventário integral do acervo",
        tcct.inventariar_dados_brutos_completo,
        tcct.CFG.dados,
    )
else:
    print("Passo 04 não solicitado.")


## Passo 05 — Auditoria estrita das entradas

**Objetivo.** Confirmar que todas as entradas obrigatórias são únicas, legíveis e compatíveis com o experimento.

**Por que existe.** O pipeline deve falhar cedo quando há arquivo ausente, ambíguo ou com dimensão incorreta, em vez de produzir resultados aparentemente plausíveis.

**Entradas críticas.** DSM, DTM, duas cenas Landsat completas, quatro `.grille`, 16 SLCs e produtos Ha-P.

**Critério para avançar.** A tabela final deve indicar `GRILLE=True` para quatro arquivos e `SLC=True` para 16 arquivos.

In [ ]:
# PASSO 05 — AUDITORIA ESTRITA DAS ENTRADAS
EXECUTAR_PASSO_05 = False
if EXECUTAR_PASSO_05:
    AUDITORIA_ENTRADAS = tcct.executar_etapa(
        "Passo 05 — Auditoria estrita das entradas",
        tcct.auditar_ambiente,
        tcct.CFG.dados, True,
    )
else:
    print("Passo 05 não solicitado.")


## Passo 06 — Reconstrução e validação do CHM/DTM

**Objetivo.** Alinhar o DTM à grade do DSM e reconstruir o CHM como `DSM − DTM`.

**Por que existe.** O CHM original não estava disponível como produto separado. A reconstrução precisa preservar CRS, resolução, NoData e alinhamento pixel a pixel.

**Saídas.** DTM alinhado, CHM bruto, CHM validado e máscara de validade.

**Critério para avançar.** O CHM deve ter shape 4384 × 2996, EPSG:32622 e valores fisicamente plausíveis, sem usar métricas do modelo para corrigir o datum.

In [ ]:
# PASSO 06 — RECONSTRUÇÃO/VALIDAÇÃO DO CHM E DTM ALINHADO
EXECUTAR_PASSO_06 = False
if EXECUTAR_PASSO_06:
    CHM = tcct.executar_etapa(
        "Passo 06 — Reconstrução/validação do CHM e DTM alinhado",
        tcct.preparar_chm_lidar_reproduzivel,
        tcct.CFG.dados,
    )
else:
    print("Passo 06 não solicitado.")


## Passo 07 — Mosaico Landsat-5 orientado por qualidade

**Objetivo.** Combinar as cenas de 29/08/2009 e 14/09/2009 usando QA e qualidade radiométrica, sem consultar o CHM.

**Por que existe.** A seleção por célula reduz nuvens e NoData mantendo independência do alvo. O uso do CHM para escolher pixels ópticos causaria vazamento.

**Saídas.** Seis bandas de reflectância, NDVI e EVI, totalizando oito atributos por célula de 30 m.

**Critério para avançar.** O mosaico deve ser finito nas regiões aceitas e registrar a cena escolhida por célula.

In [ ]:
# PASSO 07 — MOSAICO LANDSAT-5 POR QUALIDADE
EXECUTAR_PASSO_07 = False
if EXECUTAR_PASSO_07:
    LANDSAT = tcct.executar_etapa(
        "Passo 07 — Mosaico Landsat-5 por qualidade",
        tcct.preparar_mosaico_landsat_qualidade,
        tcct.CFG.dados,
    )
else:
    print("Passo 07 não solicitado.")


## Passo 08 — Referência vertical do DTM para a `.grille`

**Objetivo.** Determinar se o DTM deve ser usado diretamente ou convertido de NGG1977 para altura elipsoidal GRS80.

**Por que existe.** A terceira coordenada da `.grille` representa altitude na geometria de georreferenciamento. Um erro vertical de dezenas de metros desloca a posição radar prevista.

**Método.** A decisão usa pontos ROI e a grade geoidal oficial `fr_ign_ggguy15.tif`; não utiliza CHM, erro de rede ou ajuste pós-hoc.

**Critério para avançar.** Deve ser materializado `DTM_para_grille_GRS80.tif` e registrado o modo `ngg1977_para_elipsoidal` ou `direto`.

In [ ]:
# PASSO 08 — DECISÃO VERTICAL DO DTM E MATERIALIZAÇÃO PARA `.GRILLE`
EXECUTAR_PASSO_08 = False
if EXECUTAR_PASSO_08:
    DTM_GRILLE = tcct.executar_etapa(
        "Passo 08 — Decisão vertical do DTM e materialização para `.grille`",
        tcct.preparar_dtm_para_grille,
        tcct.CFG.dados,
    )
else:
    print("Passo 08 não solicitado.")


## Passo 09 — Auditoria dos layouts SLC e Ha-P

**Objetivo.** Verificar dimensões, dtype, offset e tamanho em bytes dos binários.

**Layouts congelados.** L/0506: 18243×2600; L/0603: 16361×2600; P/0506: 10734×4000; P/0603: 9601×4000; Ha-P: 7478×4000.

**Por que existe.** Um shape incorreto reorganiza os bytes sem necessariamente gerar erro de leitura, produzindo uma imagem cientificamente inválida.

**Critério para avançar.** O tamanho esperado deve coincidir exatamente com cada arquivo.

In [ ]:
# PASSO 09 — LAYOUTS APROVADOS
EXECUTAR_PASSO_09 = False
if EXECUTAR_PASSO_09:
    LAYOUTS_SLC = tcct.executar_etapa(
        "Passo 09a — layouts SLC",
        tcct.auditar_layouts_slc,
        tcct.CFG.dados,
    )
    LAYOUT_HA = tcct.executar_etapa(
        "Passo 09b — layout Ha-P",
        tcct.auditar_layout_ha_p,
        tcct.CFG.dados,
    )
else:
    print("Passo 09 não solicitado.")


## Passo 10 — Back-geocoding dos 16 SLCs por `.grille` 3D

**Objetivo.** Mapear cada pixel da grade LiDAR para coordenadas fracionárias de linha/coluna radar e interpolar os SLCs complexos.

**Método.** Inversão estruturada por níveis altimétricos, interpolação bilinear das partes real e imaginária e máscara sem extrapolação.

**Por que existe.** A antiga aproximação por vértices ROI e KDTree não representava uma LUT de geolocalização completa.

**Critério para avançar.** Cada aquisição/polarização deve produzir raster complexo e máscara geométrica, sem clipping de coordenadas fora do SLC.

In [ ]:
# PASSO 10 — BACK-GEOCODING DOS 16 SLCS POR `.GRILLE` 3D
EXECUTAR_PASSO_10 = False
if EXECUTAR_PASSO_10:
    BACKGEO = tcct.executar_etapa(
        "Passo 10 — Back-geocoding dos 16 SLCs por `.grille` 3D",
        tcct.executar_ortorretificacao_global,
        tcct.CFG.dados,
    )
else:
    print("Passo 10 não solicitado.")


## Passo 11 — Auditoria integral do back-geocoding

**Objetivo.** Verificar cobertura, finitude, margens de borda, shapes e consistência dos 16 produtos.

**Por que existe.** Uma execução pode terminar sem exceção e ainda conter pixels extrapolados, coordenadas fora do raster ou áreas vazias.

**Critério para avançar.** Não deve haver extrapolação; os quatro vizinhos exigidos pela interpolação bilinear precisam estar dentro do SLC.

In [ ]:
# PASSO 11 — AUDITORIA INTEGRAL DO BACK-GEOCODING
EXECUTAR_PASSO_11 = False
if EXECUTAR_PASSO_11:
    AUD_BACKGEO = tcct.executar_etapa(
        "Passo 11 — Auditoria integral do back-geocoding",
        tcct.auditar_saidas_backgeocoding,
        tcct.CFG.dados,
    )
else:
    print("Passo 11 não solicitado.")


## Passo 12 — Geometria comum L/P e 0506/0603

**Objetivo.** Criar a interseção espacial em que todas as quatro geometrias estão simultaneamente válidas.

**Por que existe.** O TCCT trimodal só pode receber um patch quando L e P, mestre e escrava, correspondem à mesma região cartográfica.

**Critério para avançar.** A máscara comum deve ser não vazia e cada patch aceito posteriormente deve ter fração geométrica 1,0.

In [ ]:
# PASSO 12 — INTERSEÇÃO GEOMÉTRICA COMUM L/P E 0506/0603
EXECUTAR_PASSO_12 = False
if EXECUTAR_PASSO_12:
    GEOMETRIA_COMUM = tcct.executar_etapa(
        "Passo 12 — Interseção geométrica comum L/P e 0506/0603",
        tcct.preparar_geometria_sar_comum,
        tcct.CFG.dados,
    )
else:
    print("Passo 12 não solicitado.")


## Passo 13 — Coerência PolInSAR normalizada

**Objetivo.** Calcular coerência complexa local para o par 0506–0603 em L e P, após o coregistro geográfico.

**Por que existe.** O produto `mestre × conj(escrava)` isolado não é coerência normalizada. A normalização precisa das potências locais de mestre e escrava.

**Saídas.** Coerência complexa, magnitude, fase, máscaras e estatísticas por banda/polarização.

**Critério para avançar.** Magnitudes válidas devem permanecer no intervalo físico [0,1], sem saturação artificial generalizada.

In [ ]:
# PASSO 13 — COERÊNCIA NORMALIZADA NA NOVA GEOMETRIA
EXECUTAR_PASSO_13 = False
if EXECUTAR_PASSO_13:
    COERENCIA = tcct.executar_etapa(
        "Passo 13 — Coerência normalizada na nova geometria",
        tcct.preparar_mapas_coerencia,
        tcct.CFG.dados,
    )
else:
    print("Passo 13 não solicitado.")


## Passo 14 — Construção do dataset v6

**Objetivo.** Gerar patches multimodais de 30 m contendo Landsat, SAR complexo L/P, coerência, qualidade, CHM e metadados espaciais.

**Por que existe.** O dataset precisa preservar coordenadas, grupo espacial, janelas LiDAR e máscaras para permitir folds auditáveis.

**Filtros.** Cobertura geométrica integral, CHM válido suficiente, altura média mínima e ausência de valores não finitos.

**Critério para avançar.** O `index.csv` deve registrar todas as amostras e seus grupos; no processamento atual foram obtidos 10.500 patches.

In [ ]:
# PASSO 14 — DATASET v6 GRILLE3D
EXECUTAR_PASSO_14 = False
CONFIRMAR_RECRIACAO_DATASET_V6 = False

if EXECUTAR_PASSO_14:
    import shutil

    if tcct.CFG.dados.dir_dataset_v2.exists():
        if not CONFIRMAR_RECRIACAO_DATASET_V6:
            raise RuntimeError(
                "O dataset v6 já existe. Revise o caminho e defina "
                "CONFIRMAR_RECRIACAO_DATASET_V6=True para recriá-lo."
            )
        shutil.rmtree(tcct.CFG.dados.dir_dataset_v2)

    tcct.CFG.dados.sobrescrever = False
    INDICE_DATASET = tcct.executar_etapa(
        "Passo 14 — dataset v6 .grille 3D",
        tcct.executar_geracao_dataset_v2,
        tcct.CFG.dados,
    )
else:
    print("Passo 14 não solicitado.")


## Passo 15 — Auditoria do dataset v6

**Objetivo.** Validar shapes, faixas, coerências, CHM, Landsat e cobertura SAR antes de qualquer treinamento.

**Por que existe.** Treinar uma rede não é método de validação de dados. Erros devem ser detectados por estatísticas e máscaras independentes.

**Critério para avançar.** Fração geométrica igual a 1,0 nos patches aceitos, valores finitos e distribuição do alvo sem colapso. `physics_valid=False` neste ponto é esperado, pois o overlay RVoG é preparado no Passo 16.

In [ ]:
# PASSO 15 — AUDITORIA DO DATASET v6
EXECUTAR_PASSO_15 = False
if EXECUTAR_PASSO_15:
    DATASET = tcct.TropiSARDatasetV2(tcct.CFG.dados.dir_dataset_v2)
    AUD_DATASET = tcct.executar_etapa(
        "Passo 15 — auditoria dataset v6",
        tcct.auditar_dataset_v2,
        DATASET,
        1500,
    )
else:
    print("Passo 15 não solicitado.")


## Passo 16 — Geometria RVoG-P derivada de Ha real

**Objetivo.** Calcular o número de onda vertical do par 0506–0603 a partir dos produtos Ha com referência comum 0007.

**Fórmula.** `kz(0506,0603) = 2π/Ha(0007,0603) − 2π/Ha(0007,0506)`. A referência 0007 cancela-se algebricamente e não entra como modalidade do TCCT.

**Calibração.** Apenas o sinal de `kz` e a atenuação efetiva são escolhidos usando o treino do fold. A validação não participa.

**Critério para avançar.** `success=true`, física válida suficiente em treino/validação e loss física restrita à Banda P.

In [ ]:
# PASSO 16 — GEOMETRIA RVoG-P
EXECUTAR_PASSO_16 = False
if EXECUTAR_PASSO_16:
    DATASET = tcct.TropiSARDatasetV2(tcct.CFG.dados.dir_dataset_v2)
    FOLDS = tcct.criar_folds_espaciais(DATASET)
    TREINO_0, VALIDACAO_0 = FOLDS[0]
    GEOMETRIA_P = tcct.executar_etapa(
        "Passo 16 — geometria RVoG-P",
        tcct.preparar_geometria_rvog,
        DATASET,
        TREINO_0,
        VALIDACAO_0,
        tcct.CFG.dados.dir_resultados_v2 / "preflight_fold_0",
    )
else:
    print("Passo 16 não solicitado.")


## Passo 17 — Auditoria espacial e sensibilidade ao buffer

**Objetivo.** Demonstrar que treino e validação não compartilham grupos e quantificar o efeito de remover blocos vizinhos.

**Protocolo principal.** GroupKFold com blocos de 8×8 células Landsat (aprox. 240×240 m), `buffer=0`.

**Sensibilidade.** `buffer=1` é auditado separadamente; ele reduz fortemente o número de amostras e não é adotado silenciosamente.

**Critério para avançar.** Índices e grupos disjuntos, manifestos salvos e mapa dos folds disponível.

In [ ]:
# PASSO 17 — AUDITORIA ESPACIAL
EXECUTAR_PASSO_17 = False
if EXECUTAR_PASSO_17:
    DATASET = tcct.TropiSARDatasetV2(tcct.CFG.dados.dir_dataset_v2)
    AUD_ESPACIAL = tcct.executar_etapa(
        "Passo 17 — auditoria espacial",
        tcct.auditar_sensibilidade_buffer,
        DATASET,
    )
else:
    print("Passo 17 não solicitado.")


## Passo 18 — Smoke test da arquitetura e da Fase B

**Objetivo.** Confirmar gradientes, capacidade de sobreajuste, transição Fase A→B, RVoG e salvamento de checkpoints em poucas épocas.

**Por que existe.** Um treino completo não deve ser iniciado antes de provar que o modelo consegue reduzir a loss e que a parcela física é numericamente estável.

**Critério para avançar.** Redução no teste de sobreajuste, métricas finitas e conclusão das duas fases. O smoke test não é resultado publicável.

In [ ]:
# PASSO 18 — SMOKE TEST
EXECUTAR_PASSO_18 = False
if EXECUTAR_PASSO_18:
    SMOKE = tcct.executar_etapa("Passo 18 — smoke test", tcct.executar_smoke_test_revisao, 0)
else:
    print("Passo 18 não solicitado.")


## Passo 19 — Treino completo do Fold 0

**Objetivo.** Executar um fold inteiro antes do experimento multiseed para verificar tempo, early stopping e seleção entre Fase A e Fase B.

**Interpretação.** A Fase B começa do melhor checkpoint da Fase A, mas otimiza uma função de perda diferente. Se não melhorar o RMSE de validação, o checkpoint supervisionado permanece como modelo final.

**Critério para avançar.** Treino concluído sem instabilidade e arquivos `comparacao_fases.csv`, `historico_completo.csv`, previsões e checkpoint final.

In [ ]:
# PASSO 19 — FOLD 0 COMPLETO
EXECUTAR_PASSO_19 = False
if EXECUTAR_PASSO_19:
    RESULTADO_FOLD0 = tcct.executar_etapa("Passo 19 — fold 0", tcct.treinar_holdout, "tcct", "completo", 0)
else:
    print("Passo 19 não solicitado.")


## Passo 20 — TCCT completo: 5 folds × 3 seeds, com retomada

**Objetivo.** Estimar generalização espacial e variabilidade de inicialização em 15 execuções pareadas.

**Novidades da v3.2.1.**
- A saída informa `seed`, `fold`, posição na fila, concluídos, pendentes e ETA.
- Combinações com `resumo_treinamento.json` e checkpoint final válidos são puladas.
- Cada época salva modelo atual, melhor modelo, otimizador, scheduler, histórico, gerador do DataLoader e estados aleatórios.
- Uma interrupção no meio da época refaz somente essa época.

**Retomada.** Após desligar ou mudar de computador, execute os Passos 00 e 01 e rode novamente o Passo 20. A primeira combinação interrompida pela versão antiga pode reiniciar uma única vez; as combinações já concluídas são preservadas.

**Uso científico.** Este é o protocolo principal para média e desvio-padrão. Não substitua por um split aleatório 70/20/10.

In [ ]:
# PASSO 20 — TCCT 5-FOLD × 3 SEEDS — RETOMÁVEL
# Primeiro, mostra o que já foi concluído e o que falta.
STATUS_PASSO20 = tcct.status_5fold_multiseed(
    "tcct",
    "completo",
)

EXECUTAR_PASSO_20 = False

if EXECUTAR_PASSO_20:
    RESULTADOS_TCCT = tcct.executar_etapa(
        "Passo 20 — TCCT multiseed retomável",
        tcct.executar_5fold_multiseed,
        "tcct",
        "completo",
    )
else:
    print(
        "Passo 20 não solicitado. Para retomar, altere "
        "EXECUTAR_PASSO_20 = True. Execuções concluídas serão puladas."
    )


 seed  fold    status      fase_selecionada  rmse_patch
   42     0 concluido fase_A_supervisionada    3.786146
   42     1 concluido fase_A_supervisionada    3.465003
   42     2 concluido fase_A_supervisionada    3.618430
   42     3 concluido fase_A_supervisionada    3.490989
   42     4 concluido fase_A_supervisionada    3.518595
  314     0 concluido fase_A_supervisionada    3.717313
  314     1 concluido fase_A_supervisionada    3.512885
  314     2 concluido fase_A_supervisionada    3.606902
  314     3 retomavel                               NaN
  314     4 concluido fase_A_supervisionada    3.564303
 2026     0 concluido fase_A_supervisionada    3.722894
 2026     1 concluido fase_A_supervisionada    3.515527
 2026     2 concluido fase_A_supervisionada    3.635271
 2026     3 concluido           fase_B_rvog    3.490520
 2026     4 concluido fase_A_supervisionada    3.596586

Resumo: {'concluido': 14, 'retomavel': 1}

▶ INÍCIO — Passo 20 — TCCT multiseed retomável
  Python: /us

Estatísticas do treino:   0%|          | 0/8400 [00:00<?, ?it/s]

Baseline média: {'rmse': 3.7594804954598153, 'r2': -0.0013495999980421836, 'bias': -0.13801856451661296, 'std_pred': 3.552713678800501e-15, 'std_true': 3.7569461629881724}


<LOCAL_PATH> UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, layers)


Teste de sobreajuste:   0%|          | 0/120 [00:00<?, ?it/s]

Sobreajuste: loss inicial=0.2788 final=0.1069 redução=61.6%

FASE A — PRÉ-TREINO SUPERVISIONADO
⏭️ fase_A_supervisionada já concluída; checkpoint reutilizado (época 4, RMSE=3.503).


... (14 outputs left out)

completo/tcct/completo/fase_B_rvog ep 9:   0%|          | 0/350 [00:00<?, ?it/s]


[fase_B_rvog] Ep 009 66.2s | RMSE patch=3.567 R²=0.098 viés=0.355 | RMSE pixel=7.184 | std ratio=0.59 | lr=5.00e-06 λ=0.0180 | ⏳ sem melhora=7/8 | melhor=época 002 RMSE=3.525


completo/tcct/completo/fase_B_rvog ep 10:   0%|          | 0/350 [00:00<?, ?it/s]


[fase_B_rvog] Ep 010 64.1s | RMSE patch=3.567 R²=0.099 viés=0.257 | RMSE pixel=7.184 | std ratio=0.59 | lr=2.50e-06 λ=0.0200 | ⏳ sem melhora=8/8 | melhor=época 002 RMSE=3.525
🛑 Early stopping em fase_B_rvog, época 10; melhor época=2, RMSE=3.525.

🏆 fase_B_rvog ENCERRADA | melhor época=002 | melhor RMSE patch=3.525 | épocas sem melhora ao final=8

🏆 MODELO FINAL | fase=fase_A_supervisionada | RMSE patch=3.503 | R²=0.131 | viés=0.058

✅ seed=314 | fold=3 concluído em 8.6 min. Restantes=0; ETA da sessão≈0.00 h.

⏭️ [10/15] seed=314 | fold=4 — já concluído; pulando.

⏭️ [11/15] seed=2026 | fold=0 — já concluído; pulando.

⏭️ [12/15] seed=2026 | fold=1 — já concluído; pulando.

⏭️ [13/15] seed=2026 | fold=2 — já concluído; pulando.

⏭️ [14/15] seed=2026 | fold=3 — já concluído; pulando.

⏭️ [15/15] seed=2026 | fold=4 — já concluído; pulando.

Resumo final:
modelo ablation  n_folds  n_seeds  patch_rmse_mean  patch_rmse_std_espacial  patch_rmse_std_seed_media  patch_r2_mean  patch_r2_std_esp

## Passo 21 — Ablação amplitude-only

**Objetivo.** Quantificar o valor da informação complexa/fase em comparação com entradas reduzidas à amplitude.

**Por que existe.** A alegação de preservação de fase precisa ser sustentada por uma comparação pareada usando os mesmos folds e seeds.

**Critério para avançar.** Produzir tabela multiseed com os mesmos manifestos espaciais do TCCT completo.

In [ ]:
# PASSO 21 — ABLAÇÃO AMPLITUDE-ONLY
EXECUTAR_PASSO_21 = False
if EXECUTAR_PASSO_21:
    RESULTADOS_AMP = tcct.executar_etapa("Passo 21 — amplitude-only", tcct.executar_5fold_multiseed, "tcct", "amplitude_only")
else:
    print("Passo 21 não solicitado.")



▶ INÍCIO — Passo 21 — amplitude-only
  Python: /usr/local/bin/python3.14
  Horário: 2026-07-30T10:52:35.874899
Fold 0: treino=8399 validação=2101 grupos_val=40
Fold 1: treino=8400 validação=2100 grupos_val=40
Fold 2: treino=8400 validação=2100 grupos_val=40
Fold 3: treino=8400 validação=2100 grupos_val=41
Fold 4: treino=8401 validação=2099 grupos_val=40
Manifestos dos folds salvos em: <LOCAL_PATH>

PASSO 20 RETOMÁVEL — modelo=tcct | ablação=amplitude_only
Combinações totais: 15 | já concluídas: 6 | pendentes: 9
Ordem: (seed=42, fold=0) → (seed=42, fold=1) → (seed=42, fold=2) → (seed=42, fold=3) → (seed=42, fold=4) → (seed=314, fold=0) → (seed=314, fold=1) → (seed=314, fold=2) → (seed=314, fold=3) → (seed=314, fold=4) → (seed=2026, fold=0) → (seed=2026, fold=1) → (seed=2026, fold=2) → (seed=2026, fold=3) → (seed=2026, fold=4)

################################################################################################
▶ EXECUÇÃO [01/15] | seed=42 | fold=0 | concluídas=6 | pendentes

Estatísticas do treino:   0%|          | 0/8399 [00:00<?, ?it/s]

Baseline média: {'rmse': 4.533745634405562, 'r2': -0.013755812648463595, 'bias': 0.5281211956077531, 'std_pred': 1.4210854715202004e-14, 'std_true': 4.502881019996123}


<LOCAL_PATH> UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, layers)


Teste de sobreajuste:   0%|          | 0/120 [00:00<?, ?it/s]

Sobreajuste: loss inicial=0.3339 final=0.1331 redução=60.2%

FASE A — PRÉ-TREINO SUPERVISIONADO


... (340 outputs left out)

Extraindo kz P real por patch:   0%|          | 0/10500 [00:00<?, ?it/s]

Calibrando atenuação/signo RVoG-P:   0%|          | 0/4096 [00:00<?, ?it/s]

Validando coerência física P:   0%|          | 0/10500 [00:00<?, ?it/s]


Geometria RVoG v2.7 preparada:
{
  "source": "Ha_diferencial_real_P_0506_0603",
  "treino": 8061,
  "validacao": 2010,
  "banda_fisica": "P",
  "banda_L_na_loss_fisica": false,
  "resumo_kz": {
    "pipeline_version": "3.2.1-SIBGRAPI-REV-GRILLE3D-RETOMAVEL",
    "source": "Ha_diferencial_real_P_0007_0506_0603_geocodificado_por_grille_0007",
    "formula": "kz_0506_0603 = 2*pi/Ha_0007_0603 - 2*pi/Ha_0007_0506",
    "referencia_geometrica_cancelada": "tropi0007",
    "par_tcct": [
      "tropi0506",
      "tropi0603"
    ],
    "ha_0506": "<LOCAL_PATH>",
    "ha_0603": "<LOCAL_PATH>",
    "grille_referencia_ha": "<LOCAL_PATH>",
    "layout_ha": {
      "pipeline_version": "3.2.1-SIBGRAPI-REV-GRILLE3D-RETOMAVEL",
      "shape_escolhido": [
        7478,
        4000
      ],
      "dtype": "float32",
      "offset_bytes": 0,
      "tamanho_esperado_bytes": 119648000,
      "origem_shape": "INF_0007_0506_0603_e_tamanho_binario_aprovados",
      "arquivos": [
        {
          "arquivo":

## Passo 22 — Ablação sem coatenção

**Objetivo.** Medir o ganho do mecanismo de fusão/gating em relação a uma fusão sem coatenção.

**Por que existe.** O revisor solicitou evidência de que o bloco de coatenção contribui além dos encoders e do decoder.

**Critério para avançar.** Mesmos folds, seeds, dados, loss e protocolo de seleção do TCCT completo.

In [ ]:
# PASSO 22 — ABLAÇÃO SEM COATENÇÃO
EXECUTAR_PASSO_22 = False
if EXECUTAR_PASSO_22:
    RESULTADOS_SEM = tcct.executar_etapa("Passo 22 — sem coatenção", tcct.executar_5fold_multiseed, "tcct_sem_coatencao", "completo")
else:
    print("Passo 22 não solicitado.")



▶ INÍCIO — Passo 22 — sem coatenção
  Python: /usr/local/bin/python3.14
  Horário: 2026-07-30T17:42:41.017665
Fold 0: treino=8399 validação=2101 grupos_val=40
Fold 1: treino=8400 validação=2100 grupos_val=40
Fold 2: treino=8400 validação=2100 grupos_val=40
Fold 3: treino=8400 validação=2100 grupos_val=41
Fold 4: treino=8401 validação=2099 grupos_val=40
Manifestos dos folds salvos em: <LOCAL_PATH>

PASSO 20 RETOMÁVEL — modelo=tcct_sem_coatencao | ablação=completo
Combinações totais: 15 | já concluídas: 9 | pendentes: 6
Ordem: (seed=42, fold=0) → (seed=42, fold=1) → (seed=42, fold=2) → (seed=42, fold=3) → (seed=42, fold=4) → (seed=314, fold=0) → (seed=314, fold=1) → (seed=314, fold=2) → (seed=314, fold=3) → (seed=314, fold=4) → (seed=2026, fold=0) → (seed=2026, fold=1) → (seed=2026, fold=2) → (seed=2026, fold=3) → (seed=2026, fold=4)

⏭️ [01/15] seed=42 | fold=0 — já concluído; pulando.

⏭️ [02/15] seed=42 | fold=1 — já concluído; pulando.

⏭️ [03/15] seed=42 | fold=2 — já concluído; pu

Estatísticas do treino:   0%|          | 0/8401 [00:00<?, ?it/s]

Baseline média: {'rmse': 3.905943012343841, 'r2': -0.0015265355363922595, 'bias': -0.15249237618536832, 'std_pred': 3.552713678800501e-15, 'std_true': 3.9029651408746937}


Teste de sobreajuste:   0%|          | 0/120 [00:00<?, ?it/s]

Sobreajuste: loss inicial=0.3474 final=0.1328 redução=61.8%

FASE A — PRÉ-TREINO SUPERVISIONADO
⏭️ fase_A_supervisionada já concluída; checkpoint reutilizado (época 7, RMSE=3.505).


Extraindo kz P real por patch:   0%|          | 0/10500 [00:00<?, ?it/s]

... (329 outputs left out)

completo/tcct_sem_coatencao/completo/fase_B_rvog ep 12:   0%|          | 0/350 [00:00<?, ?it/s]


[fase_B_rvog] Ep 012 48.4s | RMSE patch=3.566 R²=0.165 viés=0.255 | RMSE pixel=7.490 | std ratio=0.73 | lr=5.00e-06 λ=0.0200 | ⏳ sem melhora=7/8 | melhor=época 005 RMSE=3.543


completo/tcct_sem_coatencao/completo/fase_B_rvog ep 13:   0%|          | 0/350 [00:00<?, ?it/s]


[fase_B_rvog] Ep 013 48.9s | RMSE patch=3.564 R²=0.166 viés=0.271 | RMSE pixel=7.484 | std ratio=0.72 | lr=2.50e-06 λ=0.0200 | ⏳ sem melhora=8/8 | melhor=época 005 RMSE=3.543
🛑 Early stopping em fase_B_rvog, época 13; melhor época=5, RMSE=3.543.

🏆 fase_B_rvog ENCERRADA | melhor época=005 | melhor RMSE patch=3.543 | épocas sem melhora ao final=8

🏆 MODELO FINAL | fase=fase_A_supervisionada | RMSE patch=3.503 | R²=0.195 | viés=0.310

✅ seed=2026 | fold=4 concluído em 27.5 min. Restantes=0; ETA da sessão≈0.00 h.

Resumo final:
            modelo ablation  n_folds  n_seeds  patch_rmse_mean  patch_rmse_std_espacial  patch_rmse_std_seed_media  patch_r2_mean  patch_r2_std_espacial  patch_r2_std_seed_media  patch_bias_mean  patch_bias_std_espacial  patch_bias_std_seed_media  patch_std_pred_mean  patch_std_pred_std_espacial  patch_std_pred_std_seed_media  patch_std_true_mean  patch_std_true_std_espacial  patch_std_true_std_seed_media  pixel_rmse_mean  pixel_rmse_std_espacial  pixel_rmse_std_s

## Passo 23 — Baselines profundos CNN e ViT

**Objetivo.** Comparar o TCCT com arquiteturas profundas alternativas sob o mesmo protocolo espacial.

**Por que existe.** Métricas isoladas não demonstram que a arquitetura proposta é necessária.

**Critério para avançar.** CNN e ViT devem usar as mesmas partições, normalização calculada apenas no treino e três seeds.

In [ ]:
# PASSO 23 — BASELINES PROFUNDOS CNN E VIT
EXECUTAR_PASSO_23 = False
if EXECUTAR_PASSO_23:
    RESULTADOS_CNN = tcct.executar_5fold_multiseed("cnn", "completo")
    RESULTADOS_VIT = tcct.executar_5fold_multiseed("vit", "completo")
else:
    print("Passo 23 não solicitado.")


Fold 0: treino=8399 validação=2101 grupos_val=40
Fold 1: treino=8400 validação=2100 grupos_val=40
Fold 2: treino=8400 validação=2100 grupos_val=40
Fold 3: treino=8400 validação=2100 grupos_val=41
Fold 4: treino=8401 validação=2099 grupos_val=40
Manifestos dos folds salvos em: <LOCAL_PATH>

PASSO 20 RETOMÁVEL — modelo=cnn | ablação=completo
Combinações totais: 15 | já concluídas: 0 | pendentes: 15
Ordem: (seed=42, fold=0) → (seed=42, fold=1) → (seed=42, fold=2) → (seed=42, fold=3) → (seed=42, fold=4) → (seed=314, fold=0) → (seed=314, fold=1) → (seed=314, fold=2) → (seed=314, fold=3) → (seed=314, fold=4) → (seed=2026, fold=0) → (seed=2026, fold=1) → (seed=2026, fold=2) → (seed=2026, fold=3) → (seed=2026, fold=4)

################################################################################################
▶ EXECUÇÃO [01/15] | seed=42 | fold=0 | concluídas=0 | pendentes incluindo esta=15 | ETA ainda desconhecida
##########################################################################

Estatísticas do treino:   0%|          | 0/8399 [00:00<?, ?it/s]

Baseline média: {'rmse': 4.533745634405562, 'r2': -0.013755812648463595, 'bias': 0.5281211956077531, 'std_pred': 1.4210854715202004e-14, 'std_true': 4.502881019996123}


Teste de sobreajuste:   0%|          | 0/120 [00:00<?, ?it/s]

Sobreajuste: loss inicial=0.3359 final=0.1147 redução=65.8%

BASELINE CNN — TREINO SUPERVISIONADO


completo/cnn/completo/supervisionado ep 1:   0%|          | 0/349 [00:00<?, ?it/s]

... (1,182 outputs left out)

completo/vit/completo/supervisionado ep 23:   0%|          | 0/350 [00:00<?, ?it/s]


[supervisionado] Ep 023 43.1s | RMSE patch=3.657 R²=0.122 viés=0.218 | RMSE pixel=7.441 | std ratio=0.71 | lr=2.50e-05 λ=0.0000 | ⏳ sem melhora=7/8 | melhor=época 016 RMSE=3.593


completo/vit/completo/supervisionado ep 24:   0%|          | 0/350 [00:00<?, ?it/s]


[supervisionado] Ep 024 51.0s | RMSE patch=3.663 R²=0.119 viés=0.223 | RMSE pixel=7.445 | std ratio=0.70 | lr=1.25e-05 λ=0.0000 | ⏳ sem melhora=8/8 | melhor=época 016 RMSE=3.593
🛑 Early stopping em supervisionado, época 24; melhor época=16, RMSE=3.593.

🏆 supervisionado ENCERRADA | melhor época=016 | melhor RMSE patch=3.593 | épocas sem melhora ao final=8

🏆 MODELO FINAL | fase=supervisionado | RMSE patch=3.593 | R²=0.152 | viés=0.303

✅ seed=2026 | fold=4 concluído em 17.6 min. Restantes=0; ETA da sessão≈0.00 h.

Resumo final:
modelo ablation  n_folds  n_seeds  patch_rmse_mean  patch_rmse_std_espacial  patch_rmse_std_seed_media  patch_r2_mean  patch_r2_std_espacial  patch_r2_std_seed_media  patch_bias_mean  patch_bias_std_espacial  patch_bias_std_seed_media  patch_std_pred_mean  patch_std_pred_std_espacial  patch_std_pred_std_seed_media  patch_std_true_mean  patch_std_true_std_espacial  patch_std_true_std_seed_media  pixel_rmse_mean  pixel_rmse_std_espacial  pixel_rmse_std_seed_media

## Passo 24 — Random Forest

**Objetivo.** Fornecer um baseline clássico com atributos agregados de Landsat, SAR e coerência.

**Por que existe.** Um modelo tabular forte ajuda a verificar se o ganho do TCCT vem de representação espacial/complexa ou apenas de estatísticas simples.

**Critério para avançar.** Avaliação nos mesmos folds e seeds, sem acessar a validação durante o ajuste.

In [ ]:
# PASSO 24 — RANDOM FOREST
EXECUTAR_PASSO_24 = False
if EXECUTAR_PASSO_24:
    RESULTADOS_RF = tcct.executar_etapa("Passo 24 — RF", tcct.executar_rf_5fold_multiseed)
else:
    print("Passo 24 não solicitado.")



▶ INÍCIO — Passo 24 — RF
  Python: /usr/local/bin/python3.14
  Horário: 2026-07-31T06:58:21.085357
Fold 0: treino=8399 validação=2101 grupos_val=40
Fold 1: treino=8400 validação=2100 grupos_val=40
Fold 2: treino=8400 validação=2100 grupos_val=40
Fold 3: treino=8400 validação=2100 grupos_val=41
Fold 4: treino=8401 validação=2099 grupos_val=40


Features RF:   0%|          | 0/8399 [00:00<?, ?it/s]

Features RF:   0%|          | 0/2101 [00:00<?, ?it/s]

Random Forest: {'rmse': 3.799656120448675, 'r2': 0.2879546680526045, 'bias': 0.3469426812357579, 'std_pred': 1.8876306346427085, 'std_true': 4.502881019996123}


Features RF:   0%|          | 0/8400 [00:00<?, ?it/s]

Features RF:   0%|          | 0/2100 [00:00<?, ?it/s]

... (36 outputs left out)

Random Forest: {'rmse': 3.5831473486891787, 'r2': 0.09038127398397122, 'bias': -0.045913224271456596, 'std_pred': 1.4392865603983445, 'std_true': 3.7569461629881724}


Features RF:   0%|          | 0/8401 [00:00<?, ?it/s]

Features RF:   0%|          | 0/2099 [00:00<?, ?it/s]

Random Forest: {'rmse': 3.526008788073113, 'r2': 0.18383599762634728, 'bias': -0.20413360321768015, 'std_pred': 1.9447664332857433, 'std_true': 3.9029651408746937}

✓ FIM — Passo 24 — RF | duração: 357.6 s


## Passo 25 — Consolidação da tabela da revisão

**Objetivo.** Reunir métricas por fold, média espacial, desvio-padrão entre folds e instabilidade média entre seeds.

**Por que existe.** O artigo não deve misturar variação espacial com variação de inicialização nem reportar um único melhor resultado.

**Critério para avançar.** A tabela deve apontar número de folds, seeds, RMSE/R²/viés e protocolo de cada modelo.

In [ ]:
# PASSO 25 — CONSOLIDAR TABELA DA REVISÃO
EXECUTAR_PASSO_25 = False
if EXECUTAR_PASSO_25:
    TABELA = tcct.executar_etapa("Passo 25 — tabela", tcct.consolidar_tabela_revisao)
else:
    print("Passo 25 não solicitado.")



▶ INÍCIO — Passo 25 — tabela
  Python: <LOCAL_PATH>
  Horário: 2026-07-31T15:45:10.131325
                              Model  RMSE (m) ↓          R² ↑ Bias (m; closer to 0) Std. prediction (m)  Folds  Seeds per fold
TCCT (complex phase + co-attention) 3.58 ± 0.10 0.220 ± 0.073           0.17 ± 0.09         2.46 ± 0.44      5               3
              TCCT (amplitude only) 3.68 ± 0.15 0.179 ± 0.060          -0.01 ± 0.23         2.17 ± 0.34      5               3
          TCCT without co-attention 3.57 ± 0.10 0.225 ± 0.073           0.21 ± 0.18         2.49 ± 0.42      5               3
                   CNN early fusion 3.61 ± 0.11 0.210 ± 0.080           0.23 ± 0.05         2.48 ± 0.50      5               3
    Vision Transformer early fusion 3.59 ± 0.11 0.219 ± 0.072           0.17 ± 0.13         2.56 ± 0.41      5               3
                      Random Forest 3.61 ± 0.13 0.209 ± 0.077           0.00 ± 0.21         1.78 ± 0.26      5               3

✓ FIM — Passo 25 — 

## Passo 26 — Consolidação do gating modal/XAI

**Objetivo.** Agregar os pesos reais do gating/coatenção por modalidade.

**Por que existe.** Normas de ativação não devem ser descritas como atenção. Esta etapa usa o sinal efetivamente produzido pelo mecanismo de fusão.

**Critério para avançar.** Resultados por fold/seed, distribuição dos pesos e nomenclatura consistente no artigo.

In [ ]:
# PASSO 26 — CONSOLIDAR GATING MODAL
EXECUTAR_PASSO_26 = False
if EXECUTAR_PASSO_26:
    GATING = tcct.executar_etapa("Passo 26 — gating", tcct.consolidar_gating_modalidade)
else:
    print("Passo 26 não solicitado.")



▶ INÍCIO — Passo 26 — gating
  Python: <LOCAL_PATH>
  Horário: 2026-07-31T15:47:19.013569

✓ FIM — Passo 26 — gating | duração: 1.1 s


## Passo 27 — Fine-tuning espacial seguro

**Objetivo.** Avaliar adaptação local mantendo um teste final intocado.

**Estrutura.** Treino global, validação global, domínio local de ajuste, validação local e teste final separados espacialmente, com buffers onde aplicável.

**Por que existe.** O fine-tuning não pode usar o mesmo bloco para adaptar e avaliar.

**Critério para avançar.** O teste final não participa de normalização, early stopping, calibração RVoG ou escolha de hiperparâmetros.

In [ ]:
# PASSO 27 — FINE-TUNING ESPACIAL SEGURO
EXECUTAR_PASSO_27 = Falso
if EXECUTAR_PASSO_27:
    FINETUNING = tcct.executar_etapa("Passo 27 — fine-tuning", tcct.executar_finetuning_espacial_seguro)
else:
    print("Passo 27 não solicitado.")



▶ INÍCIO — Passo 27 — fine-tuning
  Python: <LOCAL_PATH>
  Horário: 2026-07-31T15:47:45.262543


Estatísticas do treino:   0%|          | 0/6250 [00:00<?, ?it/s]

Baseline média: {'rmse': 3.6220735496158127, 'r2': -0.0004977023171717221, 'bias': 0.0807856173964711, 'std_pred': 0.0, 'std_true': 3.621172528732698}


<LOCAL_PATH> UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, layers)


Teste de sobreajuste:   0%|          | 0/120 [00:00<?, ?it/s]

Sobreajuste: loss inicial=0.3433 final=0.1350 redução=60.7%

FASE A — PRÉ-TREINO SUPERVISIONADO


... (95 outputs left out)

fine_tuning_local/tcct/completo/fine_tuning_local ep 20:   0%|          | 0/42 [00:00<?, ?it/s]


[fine_tuning_local] Ep 020 5.7s | RMSE patch=3.885 R²=-0.291 viés=0.944 | RMSE pixel=7.905 | std ratio=0.59 | lr=1.25e-06 λ=0.0000 | ⏳ sem melhora=19/20 | melhor=época 001 RMSE=3.770


fine_tuning_local/tcct/completo/fine_tuning_local ep 21:   0%|          | 0/42 [00:00<?, ?it/s]


[fine_tuning_local] Ep 021 5.7s | RMSE patch=3.885 R²=-0.291 viés=0.946 | RMSE pixel=7.904 | std ratio=0.59 | lr=1.25e-06 λ=0.0000 | ⏳ sem melhora=20/20 | melhor=época 001 RMSE=3.770
🛑 Early stopping em fine_tuning_local, época 21; melhor época=1, RMSE=3.770.

🏆 fine_tuning_local ENCERRADA | melhor época=001 | melhor RMSE patch=3.770 | épocas sem melhora ao final=20
Teste final após fine-tuning: {'rmse': 2.911345273248562, 'r2': 0.08866095685071163, 'bias': -0.029105538489834576, 'std_pred': 1.239452217601881, 'std_true': 3.0496761665869725}

✓ FIM — Passo 27 — fine-tuning | duração: 975.6 s


## Passo 27A — Comparação Segura Pré/Pós Fine-Tuning

In [9]:
# ============================================================================
# PASSO 27A — COMPARAÇÃO SEGURA PRÉ/PÓS FINE-TUNING
# Não treina nem modifica modelos. Apenas avalia checkpoints já congelados.
# ============================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset


device = tcct.DEVICE

pasta_protocolo = (
    tcct.CFG.dados.dir_resultados_v2
    / "protocolo_finetuning"
)

dataset = tcct.TropiSARDatasetV2(
    tcct.CFG.dados.dir_dataset_v2
)


def carregar_indices(nome_particao):
    caminho = pasta_protocolo / f"manifesto_{nome_particao}.csv"

    if not caminho.exists():
        raise FileNotFoundError(
            f"Manifesto não encontrado: {caminho}"
        )

    tabela = pd.read_csv(caminho)

    if "dataset_index" not in tabela.columns:
        raise KeyError(
            f"A coluna dataset_index não existe em {caminho}"
        )

    return tabela["dataset_index"].to_numpy(dtype=int)


idx_ajuste = carregar_indices("local_ajuste")
idx_validacao = carregar_indices("local_validacao")
idx_teste = carregar_indices("teste_final")


def criar_loader(indices):
    return DataLoader(
        Subset(dataset, list(map(int, indices))),
        batch_size=tcct.CFG.treino.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=tcct.PIN_MEMORY,
    )


loader_ajuste = criar_loader(idx_ajuste)
loader_validacao = criar_loader(idx_validacao)
loader_teste = criar_loader(idx_teste)


checkpoint_global = (
    pasta_protocolo
    / "modelo_global"
    / "tcct"
    / "completo"
    / "melhor_modelo_final.pth"
)

checkpoint_finetune = (
    pasta_protocolo
    / "fine_tuning_local"
    / "melhor_modelo_finetune.pth"
)


def carregar_modelo(caminho_checkpoint):
    if not caminho_checkpoint.exists():
        raise FileNotFoundError(caminho_checkpoint)

    checkpoint = torch.load(
        caminho_checkpoint,
        map_location="cpu",
        weights_only=True,
    )

    stats = tcct.StatsTreino(**checkpoint["stats"])

    modelo = tcct.TCCTv2(
        tcct.CFG.modelo,
        stats,
    ).to(device)

    modelo.load_state_dict(
        checkpoint["model_state_dict"]
    )

    modelo.eval()

    return modelo, stats


modelo_global, stats_global = carregar_modelo(
    checkpoint_global
)

modelo_finetune, stats_finetune = carregar_modelo(
    checkpoint_finetune
)


def avaliar(modelo, stats, loader):
    metricas_patch, metricas_pixel, _, y_true, y_pred = (
        tcct.avaliar_modelo(
            modelo,
            loader,
            stats,
            device,
            ablation="completo",
            xai=False,
        )
    )

    return (
        metricas_patch,
        metricas_pixel,
        np.asarray(y_true),
        np.asarray(y_pred),
    )


# Modelo global antes da adaptação
met_global_teste, pix_global_teste, y_teste, pred_global_teste = (
    avaliar(
        modelo_global,
        stats_global,
        loader_teste,
    )
)

met_global_val, _, y_val, pred_global_val = avaliar(
    modelo_global,
    stats_global,
    loader_validacao,
)

# Modelo após fine-tuning
met_finetune_teste, pix_finetune_teste, y_teste_ft, pred_finetune_teste = (
    avaliar(
        modelo_finetune,
        stats_finetune,
        loader_teste,
    )
)

met_finetune_val, _, y_val_ft, pred_finetune_val = avaliar(
    modelo_finetune,
    stats_finetune,
    loader_validacao,
)

# Confirma que os dois modelos foram avaliados nos mesmos alvos
np.testing.assert_allclose(
    y_teste,
    y_teste_ft,
    rtol=0,
    atol=1e-6,
)

np.testing.assert_allclose(
    y_val,
    y_val_ft,
    rtol=0,
    atol=1e-6,
)

# Predições do modelo global no conjunto permitido para ajuste
_, _, y_ajuste, pred_global_ajuste = avaliar(
    modelo_global,
    stats_global,
    loader_ajuste,
)

# Correção simples de viés estimada somente no conjunto de ajuste
correcao_local = float(
    np.mean(y_ajuste - pred_global_ajuste)
)

pred_global_corrigida = (
    pred_global_teste + correcao_local
)

met_global_corrigido = tcct.metricas_np(
    y_teste,
    pred_global_corrigida,
)

# Baselines constantes
pred_media_global = np.full_like(
    y_teste,
    stats_global.target_mean,
)

pred_media_local = np.full_like(
    y_teste,
    np.mean(y_ajuste),
)

met_media_global = tcct.metricas_np(
    y_teste,
    pred_media_global,
)

met_media_local = tcct.metricas_np(
    y_teste,
    pred_media_local,
)


def preparar_linha(nome, metricas):
    linha = {
        "Model": nome,
        **metricas,
    }

    std_true = metricas.get("std_true", np.nan)
    std_pred = metricas.get("std_pred", np.nan)

    linha["std_ratio"] = (
        float(std_pred / std_true)
        if np.isfinite(std_true) and std_true > 0
        else np.nan
    )

    return linha


comparacao_teste = pd.DataFrame([
    preparar_linha(
        "Global checkpoint before fine-tuning",
        met_global_teste,
    ),
    preparar_linha(
        "Global checkpoint + local bias correction",
        met_global_corrigido,
    ),
    preparar_linha(
        "Model after local fine-tuning",
        met_finetune_teste,
    ),
    preparar_linha(
        "Constant global-training mean",
        met_media_global,
    ),
    preparar_linha(
        "Constant local-adaptation mean",
        met_media_local,
    ),
])

comparacao_validacao = pd.DataFrame([
    preparar_linha(
        "Global checkpoint before fine-tuning",
        met_global_val,
    ),
    preparar_linha(
        "Model after local fine-tuning",
        met_finetune_val,
    ),
])

pasta_saida = (
    pasta_protocolo
    / "diagnostico_pre_pos"
)

pasta_saida.mkdir(
    parents=True,
    exist_ok=True,
)

comparacao_teste.to_csv(
    pasta_saida / "comparacao_teste_final_pre_pos.csv",
    index=False,
)

comparacao_validacao.to_csv(
    pasta_saida / "comparacao_validacao_local_pre_pos.csv",
    index=False,
)

np.savez(
    pasta_saida / "predicoes_teste_pre_pos.npz",
    y_true=y_teste,
    pred_global=pred_global_teste,
    pred_global_corrigida=pred_global_corrigida,
    pred_finetune=pred_finetune_teste,
)

print("\nCOMPARAÇÃO NO MESMO TESTE FINAL")
print(
    comparacao_teste.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\nCOMPARAÇÃO NA MESMA VALIDAÇÃO LOCAL")
print(
    comparacao_validacao.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print(
    "\nCorreção local de viés estimada somente no conjunto "
    f"de ajuste: {correcao_local:.6f} m"
)

print(
    "\nArquivos salvos em:",
    pasta_saida,
)

<LOCAL_PATH> UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, layers)



COMPARAÇÃO NO MESMO TESTE FINAL
                                    Model     rmse        r2      bias  std_pred  std_true  std_ratio
     Global checkpoint before fine-tuning 2.953321  0.062192 -0.109686  1.051843  3.049676   0.344903
Global checkpoint + local bias correction 3.170964 -0.081123 -1.159716  1.051843  3.049676   0.344903
            Model after local fine-tuning 2.911345  0.088661 -0.029106  1.239452  3.049676   0.406421
            Constant global-training mean 3.186746 -0.091911 -0.924566  0.000000  3.049676   0.000000
           Constant local-adaptation mean 3.584008 -0.381117 -1.882708  0.000000  3.049676   0.000000

COMPARAÇÃO NA MESMA VALIDAÇÃO LOCAL
                               Model     rmse        r2     bias  std_pred  std_true  std_ratio
Global checkpoint before fine-tuning 3.647047 -0.137963 0.468510  1.650885  3.418828   0.482880
       Model after local fine-tuning 3.770472 -0.216289 0.888593  1.725397  3.418828   0.504675

Correção local de viés estima

## Passo 27B — Regeneração das figuras em Inglês

In [ ]:
# ==============================================
# PASSO 27B — REGENERAÇÃO DAS FIGURAS EM INGLÊS
# ==============================================

tcct.executar_etapa(
    "Regeneração das figuras em inglês",
    tcct.regenerar_figuras_revisao_ingles,
)


▶ INÍCIO — Regeneração das figuras em inglês
  Python: <LOCAL_PATH>
  Horário: 2026-08-01T00:08:42.952251
Fold 0: treino=8399 validação=2101 grupos_val=40
Fold 1: treino=8400 validação=2100 grupos_val=40
Fold 2: treino=8400 validação=2100 grupos_val=40
Fold 3: treino=8400 validação=2100 grupos_val=41
Fold 4: treino=8401 validação=2099 grupos_val=40
Fold 0: treino=1590 validação=2101 grupos_val=40
Fold 1: treino=1528 validação=2100 grupos_val=40
Fold 2: treino=1489 validação=2100 grupos_val=40
Fold 3: treino=1915 validação=2100 grupos_val=41
Fold 4: treino=1394 validação=2099 grupos_val=40
Cache Landsat semanticamente compatível reutilizado: 3.0.1-SIBGRAPI-REV-SEQUENCIAL

Regenerated 122 English-language figure(s):
 - <LOCAL_PATH>
 - <LOCAL_PATH>
 - <LOCAL_PATH>
 - <LOCAL_PATH>
 - <LOCAL_PATH>
 - <LOCAL_PATH>
 - <LOCAL_PATH>
 - <LOCAL_PATH>
 - <LOCAL_PATH>
 - <LOCAL_PATH>
 - <LOCAL_PATH>
 - <LOCAL_PATH>
 - <LOCAL_PATH>
 - <LOCAL_PATH>
 - <LOCAL_PATH>
 - <LOCAL_PATH>
 - <LOCAL_PATH>
 - 

[WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPa

## Passo 27C — Regeneração das figuras em Inglês com cores mais vivas

In [ ]:
# ==============================================
# PASSO 27C — REGENERAÇÃO DAS FIGURAS EM INGLÊS
# Cores mais vivas para o artigo final
# ==============================================

tcct.executar_etapa(
    "Regeneração das figuras em inglês — V2",
    tcct.regenerar_figuras_revisao_ingles,
)


▶ INÍCIO — Regeneração das figuras em inglês — V2
  Python: <LOCAL_PATH>
  Horário: 2026-08-01T12:21:57.944007
Fold 0: treino=8399 validação=2101 grupos_val=40
Fold 1: treino=8400 validação=2100 grupos_val=40
Fold 2: treino=8400 validação=2100 grupos_val=40
Fold 3: treino=8400 validação=2100 grupos_val=41
Fold 4: treino=8401 validação=2099 grupos_val=40
Fold 0: treino=1590 validação=2101 grupos_val=40
Fold 1: treino=1528 validação=2100 grupos_val=40
Fold 2: treino=1489 validação=2100 grupos_val=40
Fold 3: treino=1915 validação=2100 grupos_val=41
Fold 4: treino=1394 validação=2099 grupos_val=40
{
  "total": 10500,
  "treino": 7813,
  "validacao": 1778,
  "teste": 909,
  "fracao_treino": 0.7440952380952381,
  "fracao_validacao": 0.16933333333333334,
  "fracao_teste": 0.08657142857142858,
  "blocos_treino": [
    308,
    309,
    310,
    311,
    312,
    313,
    314,
    315,
    316
  ],
  "blocos_validacao": [
    317,
    318
  ],
  "blocos_teste": [
    319,
    320
  ],
  "divis

[WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPath('<LOCAL_PATH>'),
 WindowsPa

## Passo 27D — Comparações pareadas

In [13]:
# =================================
# PASSO 27D — COMPARAÇÕES PAREADAS
# =================================

tcct.executar_etapa(
    "Comparações OOF pareadas — Fase A estrita",
    tcct.gerar_comparacoes_pareadas_oof_faseA,
)


▶ INÍCIO — Comparações OOF pareadas — Fase A estrita
  Python: <LOCAL_PATH>
  Horário: 2026-08-01T12:39:30.265960
Fold 0: treino=8399 validação=2101 grupos_val=40
Fold 1: treino=8400 validação=2100 grupos_val=40
Fold 2: treino=8400 validação=2100 grupos_val=40
Fold 3: treino=8400 validação=2100 grupos_val=41
Fold 4: treino=8401 validação=2099 grupos_val=40


<LOCAL_PATH> UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, layers)
<LOCAL_PATH> UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, layers)
<LOCAL_PATH> UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, layers)
<LOCAL_PATH> UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, layers)
<LOCAL_PATH> UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, layers)
<LOCAL_PATH> UserWarning: enable_nested_


Strict Phase-A pairwise OOF figures saved in: <LOCAL_PATH>
               slug                     model     rmse       r2      bias  std_pred  std_true  std_ratio  n_oof seed_aggregation          phase                       prediction_source         index_alignment
      complete_tcct             Complete TCCT 3.414220 0.302000  0.174113  2.429824  4.086615   0.594581  10500 mean within fold strict Phase A fresh inference from frozen checkpoints canonical dataset_index
     amplitude_only       Amplitude-only TCCT 3.592768 0.227086 -0.005850  2.179372  4.086615   0.533295  10500 mean within fold strict Phase A fresh inference from frozen checkpoints canonical dataset_index
without_coattention TCCT without co-attention 3.397420 0.308852  0.237036  2.491271  4.086615   0.609617  10500 mean within fold strict Phase A fresh inference from frozen checkpoints canonical dataset_index
                cnn          Early-fusion CNN 3.489807 0.270752  0.226947  2.463213  4.086615   0.602751  10

{'metrics':                   slug                      model      rmse        r2  \
 0        complete_tcct              Complete TCCT  3.414220  0.302000   
 1       amplitude_only        Amplitude-only TCCT  3.592768  0.227086   
 2  without_coattention  TCCT without co-attention  3.397420  0.308852   
 3                  cnn           Early-fusion CNN  3.489807  0.270752   
 4                  vit         Vision Transformer  3.457588  0.284155   
 5        random_forest              Random Forest  2.597557  0.595980   
 
        bias  std_pred  std_true  std_ratio  n_oof  seed_aggregation  \
 0  0.174113  2.429824  4.086615   0.594581  10500  mean within fold   
 1 -0.005850  2.179372  4.086615   0.533295  10500  mean within fold   
 2  0.237036  2.491271  4.086615   0.609617  10500  mean within fold   
 3  0.226947  2.463213  4.086615   0.602751  10500  mean within fold   
 4  0.154585  2.552357  4.086615   0.624565  10500  mean within fold   
 5 -0.043433  2.406525  4.086615   0.

In [3]:
import subprocess

subprocess.run(
    [
        r"<LOCAL_PATH>",
        r"<LOCAL_PATH>",
        "--project-root",
        r"<LOCAL_PATH>",
    ],
    check=True,
)

CompletedProcess(args=['<LOCAL_PATH>', '<LOCAL_PATH>', '--project-root', '<LOCAL_PATH>'], returncode=0)

## Passo 27E — Diagnóstico definitivo do fine-tuning

In [ ]:
# ====================================================================================
# PASSO 27E — DIAGNÓSTICO DEFINITIVO DO FINE-TUNING
# Não treina nem modifica modelos. Apenas fará inferência nos checkpoints congelados.
# ====================================================================================

tcct.executar_etapa(
    "Diagnóstico seguro pré e pós fine-tuning",
    tcct.diagnosticar_finetuning_pre_pos,
)


▶ INÍCIO — Diagnóstico seguro pré e pós fine-tuning
  Python: <LOCAL_PATH>
  Horário: 2026-08-01T00:13:50.350701


<LOCAL_PATH> UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, layers)



COMPARISON ON THE SAME FINAL TEST
                               Model     rmse        r2      bias  std_pred  std_true  std_ratio
Global checkpoint before fine-tuning 2.953321  0.062192 -0.109686  1.051843  3.049676   0.344903
    Global checkpoint + local offset 3.170964 -0.081123 -1.159716  1.051843  3.049676   0.344903
       Model after local fine-tuning 2.911345  0.088661 -0.029106  1.239452  3.049676   0.406421
       Constant global-training mean 3.186746 -0.091911 -0.924566  0.000000  3.049676   0.000000
      Constant local-adaptation mean 3.584008 -0.381117 -1.882708  0.000000  3.049676   0.000000

COMPARISON ON THE SAME LOCAL VALIDATION SET
                               Model     rmse        r2     bias  std_pred  std_true  std_ratio
Global checkpoint before fine-tuning 3.647047 -0.137963 0.468510  1.650885  3.418828   0.482880
       Model after local fine-tuning 3.770472 -0.216289 0.888593  1.725397  3.418828   0.504675

VALIDATION-ONLY MODEL-SELECTION DECISION
{
  "sel

{'test_comparison':                                   Model      rmse        r2      bias  \
 0  Global checkpoint before fine-tuning  2.953321  0.062192 -0.109686   
 1      Global checkpoint + local offset  3.170964 -0.081123 -1.159716   
 2         Model after local fine-tuning  2.911345  0.088661 -0.029106   
 3         Constant global-training mean  3.186746 -0.091911 -0.924566   
 4        Constant local-adaptation mean  3.584008 -0.381117 -1.882708   
 
        std_pred  std_true     std_ratio  
 0  1.051843e+00  3.049676  3.449031e-01  
 1  1.051843e+00  3.049676  3.449031e-01  
 2  1.239452e+00  3.049676  4.064209e-01  
 3  0.000000e+00  3.049676  0.000000e+00  
 4  3.552714e-15  3.049676  1.164948e-15  ,
 'validation_comparison':                                   Model      rmse        r2      bias  \
 0  Global checkpoint before fine-tuning  3.647047 -0.137963  0.468510   
 1         Model after local fine-tuning  3.770472 -0.216289  0.888593   
 
    std_pred  std_true  std

## Passo 28 — Holdout espacial adicional 70%/20%/10% com paciência maior

**Objetivo.** Executar um experimento complementar com todas as 10.500 amostras distribuídas em aproximadamente 70% treino, 20% validação e 10% teste final.

**Separação correta.** A divisão não é aleatória por patch. O pipeline atribui colunas inteiras de blocos espaciais a faixas contíguas: treino a oeste, validação na faixa seguinte e teste a leste. Nenhum grupo é dividido. Como todas as amostras são usadas, não há faixa-buffer neste protocolo adicional; por isso ele não substitui o 5-fold espacial.

**Paciência maior.** Por padrão, a Fase A permite até 150 épocas com paciência 20, e a Fase B até 50 épocas com paciência 15. O teste de 10% permanece intocado até a seleção final do checkpoint pela validação.

**Interpretação.** Este experimento responde se o modelo continua aprendendo com uma janela maior de treinamento. As métricas principais do artigo continuam sendo as médias e dispersões do Passo 20.

In [5]:
# PASSO 28 — HOLDOUT ESPACIAL 70/20/10 — EXPERIMENTO ADICIONAL
EXECUTAR_PASSO_28 = True

if EXECUTAR_PASSO_28:
    HOLDOUT_70_20_10 = tcct.executar_etapa(
        "Passo 28 — holdout espacial 70/20/10",
        tcct.executar_holdout_espacial_70_20_10,
        42,      # seed
        True,    # paciência maior
    )
else:
    print(
        "Passo 28 não solicitado. Execute somente após concluir os "
        "experimentos principais e congelar os hiperparâmetros."
    )


▶ INÍCIO — Passo 28 — holdout espacial 70/20/10
  Python: <LOCAL_PATH>
  Horário: 2026-08-01T00:28:45.367106
{
  "total": 10500,
  "treino": 7813,
  "validacao": 1778,
  "teste": 909,
  "fracao_treino": 0.7440952380952381,
  "fracao_validacao": 0.16933333333333334,
  "fracao_teste": 0.08657142857142858,
  "blocos_treino": [
    308,
    309,
    310,
    311,
    312,
    313,
    314,
    315,
    316
  ],
  "blocos_validacao": [
    317,
    318
  ],
  "blocos_teste": [
    319,
    320
  ],
  "divisao_por_patch_aleatorio": false,
  "grupos_divididos_entre_particoes": false,
  "buffer_blocos": 0,
  "motivo_sem_buffer": "protocolo adicional solicitado para usar todas as amostras; a avaliação principal continua sendo 5-fold espacial",
  "teste_usado_em_normalizacao": false,
  "teste_usado_em_early_stopping": false,
  "teste_usado_em_calibracao_rvog": false
}


Estatísticas do treino:   0%|          | 0/7813 [00:00<?, ?it/s]

Baseline média: {'rmse': 4.464856158439673, 'r2': -0.003300489952105412, 'bias': -0.25608324673915306, 'std_pred': 0.0, 'std_true': 4.457506240746748}

FASE A — PRÉ-TREINO SUPERVISIONADO


<LOCAL_PATH> UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, layers)


completo/tcct/completo/fase_A_supervisionada ep 1:   0%|          | 0/325 [00:00<?, ?it/s]


[fase_A_supervisionada] Ep 001 36.8s | RMSE patch=4.514 R²=-0.026 viés=0.964 | RMSE pixel=8.130 | std ratio=0.23 | lr=1.00e-04 λ=0.0000 | ⭐ NOVO MELHOR | época=001 RMSE=4.514


... (82 outputs left out)

completo/tcct/completo/fase_B_rvog ep 15:   0%|          | 0/325 [00:00<?, ?it/s]


[fase_B_rvog] Ep 015 38.4s | RMSE patch=4.442 R²=0.007 viés=0.108 | RMSE pixel=8.065 | std ratio=0.51 | lr=2.50e-06 λ=0.0200 | ⏳ sem melhora=14/15 | melhor=época 001 RMSE=4.368


completo/tcct/completo/fase_B_rvog ep 16:   0%|          | 0/325 [00:00<?, ?it/s]


[fase_B_rvog] Ep 016 37.8s | RMSE patch=4.444 R²=0.006 viés=0.213 | RMSE pixel=8.067 | std ratio=0.51 | lr=2.50e-06 λ=0.0200 | ⏳ sem melhora=15/15 | melhor=época 001 RMSE=4.368
🛑 Early stopping em fase_B_rvog, época 16; melhor época=1, RMSE=4.368.

🏆 fase_B_rvog ENCERRADA | melhor época=001 | melhor RMSE patch=4.368 | épocas sem melhora ao final=15

🏆 MODELO FINAL | fase=fase_A_supervisionada | RMSE patch=4.358 | R²=0.044 | viés=0.060

Teste final intocado (10%): {'rmse': 5.87939925632405, 'r2': 0.15638710903597874, 'bias': 1.4971636340956482, 'std_pred': 2.261733114863204, 'std_true': 6.401199304884885}

✓ FIM — Passo 28 — holdout espacial 70/20/10 | duração: 1641.2 s


## Passo 29 — Resumo dos caminhos canônicos

**Objetivo.** Confirmar onde cada produto foi gravado e evitar confusão com versões antigas.

**Uso.** Execute ao final de uma sessão ou antes de copiar o projeto entre computadores. Todos os caminhos devem existir na mesma raiz detectada no Passo 00.


In [6]:
# PASSO 29 — RESUMO
for nome, caminho in {
    "CHM": tcct.CFG.dados.caminho_chm_canonico,
    "DTM alinhado": tcct.CFG.dados.caminho_dtm_alinhado,
    "DTM para grille": tcct.CFG.dados.caminho_dtm_para_grille,
    "Landsat": tcct.CFG.dados.dir_landsat_processado / "mosaico_landsat_qualidade.npz",
    "Back-geocoding v3.2": tcct.CFG.dados.dir_backgeo_grille3d,
    "Geometria SAR comum": (
        tcct.CFG.dados.dir_geometria_sar
        / "Comum_LP_0506_0603"
        / "mascara_geometrica_comum_quatro_grilles.npy"
    ),
    "Dataset v6": tcct.CFG.dados.dir_dataset_v2,
    "Resultados v3.2": tcct.CFG.dados.dir_resultados_v2,
}.items():
    print(f"{nome:<24} existe={caminho.exists()} | {caminho}")


CHM                      existe=True | <LOCAL_PATH>
DTM alinhado             existe=True | <LOCAL_PATH>
DTM para grille          existe=True | <LOCAL_PATH>
Landsat                  existe=True | <LOCAL_PATH>
Back-geocoding v3.2      existe=True | <LOCAL_PATH>
Geometria SAR comum      existe=True | <LOCAL_PATH>
Dataset v6               existe=True | <LOCAL_PATH>
Resultados v3.2          existe=True | <LOCAL_PATH>


## Passo 30 — Cálculo da Equação Alométrica

In [ ]:
# =========================================
# PASSO 30 — CÁLCULO DA EQUAÇÃO ALOMÉTRICA
# =========================================

!<LOCAL_PATH> "<LOCAL_PATH>"

TCCT AGB — SUPPORT-MATCHED ALLOMETRIC CALIBRATION
Version: 1.0.0-support-matched
Project root: <LOCAL_PATH>
Canonical CHM: <LOCAL_PATH>
Inventory source: <LOCAL_PATH>
Output: <LOCAL_PATH>

Final support-matched equation:
AGB = 4.85620358 * H^1.29546797
LOPO: RMSE=66.346 Mg/ha | rRMSE=18.289% | R²=0.3423 | bias=-15.001 Mg/ha
Accepted cells: 472/475
Results saved in: <LOCAL_PATH>


## Passo 31 — Cálculo da AGB

In [1]:
# ==========================
# PASSO 31 — CÁLCULO DA AGB 
# ==========================

import subprocess

comando = [
    r"<LOCAL_PATH>",
    r"<LOCAL_PATH>",
    "--project-root",
    r"<LOCAL_PATH>",
]

subprocess.run(comando, check=True)

CompletedProcess(args=['<LOCAL_PATH>', '<LOCAL_PATH>', '--project-root', '<LOCAL_PATH>'], returncode=0)

In [4]:
import subprocess

subprocess.run(
    [
        r"<LOCAL_PATH>",
        r"<LOCAL_PATH>",
        "--project-root",
        r"<LOCAL_PATH>",
    ],
    check=True,
)

CompletedProcess(args=['<LOCAL_PATH>', '<LOCAL_PATH>', '--project-root', '<LOCAL_PATH>'], returncode=0)